In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
print(f"{env.observation_space=}")
print(f"{env.action_space=}")

env.observation_space=Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
env.action_space=Discrete(2)


In [5]:
obs, _ = env.reset(seed=0)
print("obs:", obs)

obs: [ 0.01369617 -0.02302133 -0.04590265 -0.04834723]


In [6]:
total, done = 0.0, False
while not done:
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
    total += r
    done = term or trunc
print("random episode reward:", total)

random episode reward: 12.0


In [7]:
scores = []
for _ in range(100):
    env.reset()
    total, done = 0.0, False
    while not done:
        _, r, term, trunc, _ = env.step(env.action_space.sample())
        total += r
        done = term or trunc
    scores.append(total)
print("random baseline:", np.mean(scores))

random baseline: 21.13


In [9]:
vis = gym.make("CartPole-v1", render_mode="human")
vis.reset(seed=0)
for _ in range(200):
    _, _, term, trunc, _ = vis.step(vis.action_space.sample())
    if term or trunc:
        vis.reset()
vis.close()

In [13]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"


def make_policy():
    return nn.Sequential(
        nn.Linear(4, 128),
        nn.ReLU(),
        nn.Linear(128, 2),  # logits, one per action
    ).to(device)


policy = make_policy()
s, _ = env.reset(seed=0)
logits = policy(torch.tensor(s, dtype=torch.float32, device=device))
dist = torch.distributions.Categorical(logits=logits)
print("logits:", logits)
print("probs :", dist.probs)
print("sample:", dist.sample().item())
print("logprob:", dist.log_prob(dist.sample()))

logits: tensor([-0.0719,  0.0369], device='cuda:0', grad_fn=<ViewBackward0>)
probs : tensor([0.4728, 0.5272], device='cuda:0', grad_fn=<SoftmaxBackward0>)
sample: 0
logprob: tensor(-0.6402, device='cuda:0', grad_fn=<SqueezeBackward1>)


In [14]:
def act(policy, s):
    """Returns (action:int, log_prob:tensor) — keep the tensor, it needs a gradient."""
    s = torch.tensor(s, dtype=torch.float32, device=device)
    logits = policy(s)
    dist = torch.distributions.Categorical(logits=logits)
    a = dist.sample()
    return a.item(), dist.log_prob(a)

In [15]:
def reinforce(n_episodes=1500, gamma=0.99, lr=1e-3, normalize=True, seed=0):
    torch.manual_seed(seed)
    env = gym.make("CartPole-v1")
    policy = make_policy()
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    log = []

    for ep in range(n_episodes):
        s, _ = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            a, lp = act(policy, s)
            s, r, term, trunc, _ = env.step(a)
            log_probs.append(lp)
            rewards.append(r)
            done = term or trunc

        G, Gs = 0.0, []
        for r in reversed(rewards):
            G = r + gamma * G
            Gs.append(G)
        Gs = list(reversed(Gs))
        Gs = torch.tensor(Gs, dtype=torch.float32, device=device)
        if normalize:
            Gs = (Gs - Gs.mean()) / (Gs.std() + 1e-8)

        loss = -(torch.stack(log_probs) * Gs).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()

        log.append(sum(rewards))
        if (ep + 1) % 50 == 0:
            print(f"ep {ep + 1:4d}  avg50={np.mean(log[-50:]):6.1f}")

    return policy, log


policy, log = reinforce()

ep   50  avg50=  18.9
ep  100  avg50=  30.6
ep  150  avg50=  41.4
ep  200  avg50=  62.5
ep  250  avg50= 117.3
ep  300  avg50= 165.9
ep  350  avg50= 213.2
ep  400  avg50= 261.2
ep  450  avg50= 305.6
ep  500  avg50= 243.3
ep  550  avg50= 311.6
ep  600  avg50= 274.7
ep  650  avg50= 407.8
ep  700  avg50= 443.6
ep  750  avg50= 441.2
ep  800  avg50= 472.1
ep  850  avg50= 442.9
ep  900  avg50= 475.2
ep  950  avg50= 471.9
ep 1000  avg50= 385.9
ep 1050  avg50= 478.3
ep 1100  avg50= 487.6
ep 1150  avg50= 485.2
ep 1200  avg50= 442.4
ep 1250  avg50= 466.2
ep 1300  avg50= 499.1
ep 1350  avg50= 488.7
ep 1400  avg50= 483.7
ep 1450  avg50= 469.9
ep 1500  avg50= 409.2


In [17]:
vis = gym.make("CartPole-v1", render_mode="human")
s, _ = vis.reset(seed=0)
total, done = 0.0, False
with torch.no_grad():
    while not done:
        logits = policy(torch.tensor(s, dtype=torch.float32, device=device))
        a = logits.argmax().item()  # greedy; use act(policy, s)[0] to sample instead
        s, r, term, trunc, _ = vis.step(a)
        total += r
        done = term or trunc
vis.close()
print("policy episode reward:", total)

policy episode reward: 500.0
